# Inspeção dos produtos SPHEREx

Selecione o kernel `.venv/bin/python`. Este notebook inventaria HDUs e metadados;
não assume que a terceira dimensão de um FITS representa tempo.

Referência: [produtos oficiais](https://irsa.ipac.caltech.edu/spherex-archive-documentation/spherex-data-products/).
Para estudar movimento, precisamos identificar exposições distintas que cubram a
mesma região, seus meios de exposição e suas respostas espectrais.

In [1]:
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
print("Projeto:", ROOT)

Projeto: /home/tatiana/Transferências/projetos/nasaSpaceApps


In [2]:
import os

from spherex_tracking.fits_io import inventory_path

# Substitua pelo diretório ou pelo arquivo FITS que deseja inspecionar.
DATA_PATH = Path(os.environ.get("SPHEREX_DATA_DIR", "data/spherex"))
if not DATA_PATH.is_absolute():
    DATA_PATH = ROOT / DATA_PATH

records = inventory_path(DATA_PATH) if DATA_PATH.exists() else []
print(f"Arquivos FITS encontrados: {len(records)}")
if not records:
    print("Sem FITS locais. Defina DATA_PATH acima e execute novamente.")

Arquivos FITS encontrados: 12


In [3]:
from IPython.display import JSON, display

for record in records[:3]:
    display(JSON(record))

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

## Antes de passar imagens ao detector

- Identificar `IMAGE`, `VARIANCE`, `FLAGS`, `ZODI` e a PSF/ePSF do release.
- Verificar unidade, convenção temporal e meio da exposição nos cabeçalhos.
- Selecionar sobreposição espacial e medir a distribuição dos intervalos temporais.
- Conferir a banda efetiva em CWAVE/CBAND na trajetória, pois o filtro é LVF.
- Definir os bits de máscara pela documentação e validar a astrometria.

Não há carregamento automático desses arquivos no detector: o núcleo atual exige
imagens já registradas e fotometricamente comparáveis. O eixo com 102 canais de
um cubo all-sky é espectral, não uma sequência de 102 exposições.